# 🧠 Fine-tune Vio's real model — free on a Colab T4

This teaches a model that **already understands language** (Qwen2.5-7B) *your* networking &
security knowledge, using a **QLoRA** (4-bit) fine-tune. It runs on a **free T4 GPU** — about
**20× faster than your laptop, $0** — and produces a model that reasons *and* knows your domain.

The result exports to **GGUF** and runs locally in **Ollama**, offline, like any other model.

**First:** top menu → **Runtime → Change runtime type → T4 GPU → Save.** Then **Runtime → Run all.**

⏱ ~1–2 hours total on a free T4 (install + train + GGUF export). Keep the tab open.


### 1) Confirm the free GPU is attached


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → T4 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0), '·', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')


### 2) Install Unsloth
Unsloth makes a 7B QLoRA fit and train fast on a 16 GB T4. (~3–5 min.)


In [ ]:
%%capture
# Unsloth auto-detects Colab. If this ever fails, use the pinned fallback in the next comment.
!pip install unsloth
# fallback: !pip install --no-deps 'git+https://github.com/unslothai/unsloth.git' && pip install unsloth_zoo


### 3) Upload your `mind` folder as a zip
On your PC: right-click the **`mind`** folder → **Send to → Compressed (zipped) folder** → `mind.zip`.
Run this, click **Choose Files**, pick `mind.zip`.


In [ ]:
import zipfile, os, glob, sys
from google.colab import files
up = files.upload()
with zipfile.ZipFile(list(up.keys())[0]) as z: z.extractall('vio_src')
hit = glob.glob('vio_src/**/data_ingest.py', recursive=True)
assert hit, 'data_ingest.py not found — zip the mind folder itself and re-upload.'
MIND = os.path.dirname(os.path.abspath(hit[0]))
sys.path.insert(0, MIND)
print('Vio code at:', MIND)


### 4) Build the question→answer training set
This turns your curated datasets into instruction pairs (the same skills Vio already builds).
More datasets → more pairs → a sharper model. You can rerun this after adding data.


In [ ]:
import re, importlib, data_ingest
importlib.reload(data_ingest)
skills_path = data_ingest.build_skills(repeat=1)   # writes corpus/skills.txt, returns path
raw = open(skills_path, encoding='utf-8').read()
pairs = [{'q':m.group(1).strip(), 'a':m.group(2).strip()}
         for m in re.finditer(r'Question:\s*(.+?)\nAnswer:\s*(.+?)(?:\n\n|\Z)', raw, re.S)]
pairs = [p for p in pairs if p['q'] and p['a']]
print(f'{len(pairs)} question→answer pairs')
print('example:', pairs[0])


### 5) Load Qwen2.5-7B (4-bit) and attach a LoRA
The base model is frozen; only the small LoRA adapters train — that's why 7B fits on a free T4.
*(Prefer Llama? swap the model name for* `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`*.)*


In [ ]:
from unsloth import FastLanguageModel
import torch
MAXLEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length = MAXLEN, dtype = None, load_in_4bit = True)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0, bias = 'none',
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth', random_state = 42)


### 6) Format with the chat template and train
3 epochs is plenty for a dataset this size. Watch the loss fall in the log.


In [ ]:
from datasets import Dataset
def fmt(ex):
    msgs = [{'role':'user','content':ex['q']}, {'role':'assistant','content':ex['a']}]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}
ds = Dataset.from_list(pairs).map(fmt)

from trl import SFTTrainer
from transformers import TrainingArguments
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    dataset_text_field = 'text', max_seq_length = MAXLEN, packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 5, num_train_epochs = 3, learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(), bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10, optim = 'adamw_8bit', weight_decay = 0.01,
        lr_scheduler_type = 'linear', seed = 42, output_dir = 'outputs', report_to = 'none'))
trainer.train()


### 7) Quick sanity check
Ask it something from your domain — it should answer in your material's voice.


In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{'role':'user','content':'What is a VDOM in FortiGate and when would I use one?'}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=ids, max_new_tokens=220, temperature=0.6)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))


### 8) Export to GGUF and download
Merges the LoRA into the base and quantizes to a single file Ollama can run. (~15–25 min.)


In [ ]:
model.save_pretrained_gguf('vio_model', tokenizer, quantization_method='q4_k_m')
import glob, os
from google.colab import files
# Unsloth may save under vio_model/ or vio_model_gguf/ — search everywhere.
cands = glob.glob('**/*Q4_K_M.gguf', recursive=True) or glob.glob('**/*.gguf', recursive=True)
assert cands, 'No .gguf produced — check the conversion log above.'
gguf = cands[0]
print('GGUF:', gguf, '·', round(os.path.getsize(gguf)/1e9,2), 'GB')
print('→ your Modelfile line:  FROM ./' + os.path.basename(gguf))
files.download(gguf)


### 9) Run it locally in Vio
On your PC, in the folder where the `.gguf` downloaded:

**a)** Create a file named `Modelfile` (no extension) containing — put the real .gguf filename:
```
FROM ./Qwen2.5-7B-Instruct.Q4_K_M.gguf
```
**b)** Register it with Ollama:
```
ollama create vio-net -f Modelfile
```
**c)** Point Vio at it and restart. In the same terminal you launch Vio from:
```
set VIO_LLM_MODEL=vio-net
python web.py
```
That's it — Vio now reasons through **your** fine-tuned model, fully local and offline.

---
**To improve it later:** add more datasets to `mind/datasets/`, then rerun this notebook from
step 4. More question→answer pairs = a sharper model. Fine-tune teaches *voice and shape*;
keep feeding facts through Vio's document library (retrieval) so they stay current.
